In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch 
import torch.nn as nn
import pandas as pd
import os
from glob import glob
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import jaccard_score, precision_score, recall_score, f1_score
import torch 
from skimage import exposure, util
from skimage.color import rgb2gray
from skimage.transform import resize
from skimage.measure import label, regionprops
from skimage.morphology import binary_opening, binary_closing, square
import scipy.io as sio

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, init_features=32):
        super(UNet, self).__init__()
        features = init_features
        self.encoder1 = self._block(in_channels, features, name="enc1")
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder2 = self._block(features, features * 2, name="enc2")
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder3 = self._block(features * 2, features * 4, name="enc3")
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder4 = self._block(features * 4, features * 8, name="enc4")
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = self._block(features * 8, features * 16, name="bottleneck")

        self.upconv4 = nn.ConvTranspose2d(
            features * 16, features * 8, kernel_size=2, stride=2
        )
        self.decoder4 = self._block((features * 8) * 2, features * 8, name="dec4")
        self.upconv3 = nn.ConvTranspose2d(
            features * 8, features * 4, kernel_size=2, stride=2
        )
        self.decoder3 = self._block((features * 4) * 2, features * 4, name="dec3")
        self.upconv2 = nn.ConvTranspose2d(
            features * 4, features * 2, kernel_size=2, stride=2
        )
        self.decoder2 = self._block((features * 2) * 2, features * 2, name="dec2")
        self.upconv1 = nn.ConvTranspose2d(
            features * 2, features, kernel_size=2, stride=2
        )
        self.decoder1 = self._block(features * 2, features, name="dec1")

        self.conv = nn.Conv2d(
            in_channels=features, out_channels=out_channels, kernel_size=1
        )
        self.sigmoid = nn.Sigmoid()

    def _block(self, in_channels, features, name):
        return nn.Sequential(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=features,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(num_features=features),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                in_channels=features,
                out_channels=features,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(num_features=features),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool1(enc1))
        enc3 = self.encoder3(self.pool2(enc2))
        enc4 = self.encoder4(self.pool3(enc3))

        bottleneck = self.bottleneck(self.pool4(enc4))

        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.decoder4(dec4)
        dec3 = self.upconv3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.decoder3(dec3)
        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.decoder2(dec2)
        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.decoder1(dec1)
        
        logits = self.conv(dec1)
        return self.sigmoid(logits)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 
model = UNet(in_channels=3, out_channels=1).to(device)
model.eval()
model_save_path = "/kaggle/input/384-best-unet/pytorch/default/1/384_unet.pth"
try:
    with torch.serialization.safe_globals([np.dtype, np.core.multiarray.scalar]):
        checkpoint = torch.load(model_save_path, map_location=device, weights_only=True)
except:
    print(" weights_only=True load failed, try weights_only=False")
    checkpoint = torch.load(model_save_path, map_location=device, weights_only=False)

model.load_state_dict(checkpoint['model_state_dict'])
print(f"success")

In [ ]:
def get_bounding_box(binary_mask):
    binary = binary_mask > 0
    if not np.any(binary):
        height, width = binary_mask.shape
        return 0, 0, width, height
    labeled_image = label(binary)
    regions = regionprops(labeled_image)
    largest_region = max(regions, key=lambda r: r.area)
    min_row, min_col, max_row, max_col = largest_region.bbox
    return min_col, min_row, max_col - min_col, max_row - min_row

def gamma_correct(img, gamma=0.4):
    img_float = util.img_as_float(img)
    corrected = exposure.adjust_gamma(img_float, gamma=gamma)
    return util.img_as_ubyte(corrected)

def enhance_contrast_clahe(image, clip_limit=0.02, kernel_size=8):
    img_float = util.img_as_float(image)
    enhanced = exposure.equalize_adapthist(
        img_float,
        kernel_size=kernel_size,
        clip_limit=clip_limit
    )
    return util.img_as_ubyte(enhanced)

def segment_optic_disc_from_mat(image_path):
    mat_dir = "/kaggle/input/glaucoma-detection/ORIGA/ORIGA/Semi-automatic-annotations/"
    image_stem = Path(image_path).stem
    mat_path = os.path.join(mat_dir, f"{image_stem}.mat")
    mat_data = sio.loadmat(mat_path)
    mask = mat_data['mask']
    optic_disc_mask = (mask > 0).astype(np.uint8) * 255
    return optic_disc_mask

def segment_optic_disc_by_clustering(gray_image):
    pixel_values = gray_image.reshape(-1, 1).astype(np.float32)
    kmeans = KMeans(n_clusters=6, n_init=3, random_state=0).fit(pixel_values)
    cluster_labels = kmeans.labels_.reshape(gray_image.shape)
    cluster_brightness_means = []
    for cluster_id in range(6):
        pixels_in_cluster = pixel_values[cluster_labels.ravel() == cluster_id]
        mean_brightness = pixels_in_cluster.mean() if len(pixels_in_cluster) > 0 else -np.inf
        cluster_brightness_means.append(mean_brightness)
    brightest_cluster_ids = np.argsort(cluster_brightness_means)[-2:]
    optic_disc_mask = np.isin(cluster_labels, brightest_cluster_ids)
    return (optic_disc_mask * 255).astype(np.uint8)

def refine_mask_with_morphology(binary_mask):
    binary = binary_mask > 0
    opened = binary_opening(binary, footprint=square(5))
    closed = binary_closing(opened, footprint=square(5))
    return (closed * 255).astype(np.uint8)

def crop_fundus_region(original_image, threshold=10):
    if original_image.ndim == 2:
        original_image = np.stack([original_image] * 3, axis=-1)
    gray_image = util.img_as_ubyte(rgb2gray(original_image))
    if np.max(gray_image) <= threshold:
        foreground_mask = np.ones_like(gray_image, dtype=np.uint8) * 255
    else:
        foreground_mask = (gray_image > threshold).astype(np.uint8) * 255
    min_col, min_row, width, height = get_bounding_box(foreground_mask)
    margin = min(100, width // 4, height // 4)
    col_start = max(0, min_col + margin)
    row_start = max(0, min_row + margin)
    col_end = min(original_image.shape[1], min_col + width - margin)
    row_end = min(original_image.shape[0], min_row + height - margin)
    if col_end <= col_start or row_end <= row_start:
        col_start, row_start, col_end, row_end = 0, 0, original_image.shape[1], original_image.shape[0]
    cropped_rgb = original_image[row_start:row_end, col_start:col_end]
    cropped_gray = gray_image[row_start:row_end, col_start:col_end]
    return cropped_rgb, cropped_gray

def segment_optic_disc_with_unet(image_path, unet_model, device='cpu'):
    pil_image = Image.open(image_path).convert('RGB')
    original_img = np.array(pil_image)
    img_tensor = util.img_as_float(original_img)
    img_tensor = resize(img_tensor, (384, 384), anti_aliasing=True)
    img_tensor = torch.FloatTensor(img_tensor).permute(2, 0, 1).unsqueeze(0)  
    img_tensor = img_tensor.to(device)
    with torch.no_grad():
        output = unet_model(img_tensor)
        mask_pred = output.squeeze().cpu().numpy()
        mask_pred = (mask_pred > 0.5).astype(np.uint8) 
    mask_pred = resize(mask_pred, original_img.shape[:2], anti_aliasing=False, order=0)
    mask_pred = (mask_pred > 0.5).astype(np.uint8) * 255
    return mask_pred

def evaluate_segmentation(pred_mask, true_mask):
    pred_binary = (pred_mask > 0).astype(np.uint8)
    true_binary = (true_mask > 0).astype(np.uint8)
    if pred_binary.shape != true_binary.shape:
        pred_binary = resize(
            pred_binary, 
            true_binary.shape, 
            anti_aliasing=False, 
            order=0 
        ).astype(np.uint8)
    iou = jaccard_score(true_binary.flatten(), pred_binary.flatten(), average='binary', zero_division=0)
    precision = precision_score(true_binary.flatten(), pred_binary.flatten(), average='binary', zero_division=0)
    recall = recall_score(true_binary.flatten(), pred_binary.flatten(), average='binary', zero_division=0)
    f1 = f1_score(true_binary.flatten(), pred_binary.flatten(), average='binary', zero_division=0)
    dice = 2 * np.sum(pred_binary & true_binary) / (np.sum(pred_binary) + np.sum(true_binary) + 1e-6)
    return {
        'IoU': iou,
        'Dice': dice,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

def compare_segmentation_methods(image_paths, unet_model=None, device='cpu'):
    results = []
    for image_path in image_paths:
        print(f"Processing {image_path}")
        pil_image = Image.open(image_path).convert("RGB")
        original_image = np.array(pil_image)
        cropped_rgb, cropped_gray = crop_fundus_region(original_image, threshold=10)
        enhanced_gray = gamma_correct(cropped_gray, gamma=0.4)
        enhanced_gray = enhance_contrast_clahe(enhanced_gray)
        true_mask = segment_optic_disc_from_mat(image_path)
        clustering_mask = segment_optic_disc_by_clustering(enhanced_gray)
        clustering_metrics = evaluate_segmentation(clustering_mask, true_mask)
        unet_metrics = {'IoU': 0, 'Dice': 0, 'Precision': 0, 'Recall': 0, 'F1-Score': 0}
        if unet_model is not None:
            unet_mask = segment_optic_disc_with_unet(image_path, unet_model, device=device)
            unet_metrics = evaluate_segmentation(unet_mask, true_mask)
        row = {
            'Filename': Path(image_path).name,
            'Clustering_IoU': clustering_metrics['IoU'],
            'Clustering_Dice': clustering_metrics['Dice'],
            'Clustering_Precision': clustering_metrics['Precision'],
            'Clustering_Recall': clustering_metrics['Recall'],
            'Clustering_F1': clustering_metrics['F1-Score'],
            'UNet_IoU': unet_metrics['IoU'],
            'UNet_Dice': unet_metrics['Dice'],
            'UNet_Precision': unet_metrics['Precision'],
            'UNet_Recall': unet_metrics['Recall'],
            'UNet_F1': unet_metrics['F1-Score']
        }
        results.append(row)
    results_df = pd.DataFrame(results)
    avg_results = results_df.mean(numeric_only=True).to_dict()
    avg_results['Filename'] = 'Average'
    results_df = pd.concat([results_df, pd.DataFrame([avg_results])], ignore_index=True)
    return results_df

In [ ]:
print("compare segment approach")
image_dir = "/kaggle/input/glaucoma-detection/ORIGA/ORIGA/Images/"
image_paths = sorted(glob(os.path.join(image_dir, "*.jpg")))
# image_paths = image_paths[:100]
results_df = compare_segmentation_methods(image_paths, unet_model=model, device=device)

print(results_df)
results_df.to_csv("/kaggle/working/compare_results.csv")

In [ ]:
print("star visualizing")
for i in range(min(3, len(image_paths))):
    image_path = image_paths[i]
    pil_image = Image.open(image_path).convert("RGB")
    original_image = np.array(pil_image)
    
    true_mask = segment_optic_disc_from_mat(image_path)
    clustering_mask = segment_optic_disc_by_clustering(rgb2gray(original_image))
    unet_mask = segment_optic_disc_with_unet(image_path, model, device=device)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(original_image)
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    
    axes[1].imshow(true_mask, cmap='gray')
    axes[1].set_title("Ground Truth (MAT)")
    axes[1].axis('off')
    
    axes[2].imshow(clustering_mask, cmap='gray')
    axes[2].set_title("Clustering")
    axes[2].axis('off')
    
    axes[3].imshow(unet_mask, cmap='gray')
    axes[3].set_title("UNet (384x384)")
    axes[3].axis('off')
    
    plt.suptitle(f"Comparison for {Path(image_path).name}")
    plt.show()